In [2]:
!pip install gradio
!pip install langchain
!pip install nltk
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.5/127.5 kB 2.6 MB/s eta 0:00:00


In [9]:
!pip install langchain-groq

In [1]:
import os
import gradio as gr
from langchain_groq import ChatGroq
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import defaultdict
import string
import nltk

# Set your Groq API key
os.environ["GROQ_API_KEY"] = "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"

# Download necessary NLTK resources
nltk.download('vader_lexicon')
nltk.download('punkt_tab',force=True)
nltk.download('stopwords')

# Sentiment Analysis Setup
reviews = [
    "Praises the taste of the food",
    "Praises about the fast delivery of food",
    "Complaints on the quantity of food"
]

sid = SentimentIntensityAnalyzer()
stop_words = set(stopwords.words('english'))
translator = str.maketrans('', '', string.punctuation)

sentiment_reviews = defaultdict(list)

for review in reviews:
    review = review.lower()
    tokens = word_tokenize(review)
    tokens = [token.translate(translator) for token in tokens if token not in stop_words]
    compound_score = sid.polarity_scores(review)['compound']
    sentiment = 'positive' if compound_score >= 0 else 'negative'
    sentiment_reviews[sentiment].append(tokens)

summary = ""

for sentiment, reviews in sentiment_reviews.items():
    summary += f"People usually {sentiment} these aspects: "
    for aspect_reviews in reviews:
        summary += ' '.join(aspect_reviews) + ", "
    summary += " "

# Prompt and LLM setup
template = """You're a customer review expert! Based on the given reviews, summarize the positives and negatives and give feedback in 2 lines.
{chat_history}
User: {user_message}
Chatbot:"""

prompt = PromptTemplate(
    input_variables=["chat_history", "user_message"], template=template
)

memory = ConversationBufferMemory(memory_key="chat_history")

llm_chain = LLMChain(
    llm=ChatGroq(temperature=0.5, model_name="llama3-70b-8192"),
    prompt=prompt,
    verbose=True,
    memory=memory,
)

# Gradio response function
def get_text_response(user_message, history):
    return llm_chain.predict(user_message=user_message)

# Launch Gradio Chat Interface
demo = gr.ChatInterface(get_text_response, examples=[
    "Pizza, Pasta, Italian Meatballs",
    "Burger, Fries, Milkshake"
])

if __name__ == "__main__":
    demo.launch(debug=True)


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
<ipython-input-1-9e750a91f41a>:61: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history")
<ipython-input-1-9e750a91f41a>:63: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(
/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:338: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gra

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2527ed8fffd08aa5ee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)




> Entering new LLMChain chain...
Prompt after formatting:
You're a customer review expert! Based on the given reviews, summarize the positives and negatives and give feedback in 2 lines.

User: 1.The crust was perfectly crispy, and the cheese just melted in my mouth. Absolutely delicious!

2.Loved the toppings variety, but I wish the pizza had arrived a bit hotter.

3.Fast delivery and great portion size. The pepperoni pizza is my favorite!

4.The sauce was too tangy for my taste, and the base felt a bit undercooked.
Chatbot:

> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
You're a customer review expert! Based on the given reviews, summarize the positives and negatives and give feedback in 2 lines.
Human: 1.The crust was perfectly crispy, and the cheese just melted in my mouth. Absolutely delicious!

2.Loved the toppings variety, but I wish the pizza had arrived a bit hotter.

3.Fast delivery and great portion size. The pepperoni pizza is my favorite!
